In [3]:
#Random Forests
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score

# Load your data (replace with your dataset)
data = pd.read_csv('student_dataset.csv')  # Make sure your CSV file has the correct columns

# Define features (X) and target (y)
X = data[['age', 'study_hours', 'attendance', 'assignments_completed']]
y = data['pass_fail']

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Create individual models
model1 = DecisionTreeClassifier()
model2 = LogisticRegression()
model3 = SVC()

# Train individual models
model1.fit(X_train, y_train)
model2.fit(X_train, y_train)
model3.fit(X_train, y_train)

# Make predictions from individual models
pred1 = model1.predict(X_test)
pred2 = model2.predict(X_test)
pred3 = model3.predict(X_test)

# Ensemble model: Simple majority voting for classification
ensemble_pred = [max(set([p1, p2, p3]), key=[p1, p2, p3].count) for p1, p2, p3 in zip(pred1, pred2, pred3)]

# Evaluate the ensemble model
accuracy = accuracy_score(y_test, ensemble_pred)
print(f"Ensemble Model Accuracy: {accuracy}")


Ensemble Model Accuracy: 0.965


In [5]:
#voting
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import VotingClassifier
from sklearn.metrics import accuracy_score
from sklearn.feature_extraction.text import TfidfVectorizer


# Load your sentiment analysis dataset (replace with your data)
data = pd.read_csv('sentiment_data.csv')
print(data)
# Assuming 'sentiment_data.csv' has columns like 'text' and 'sentiment' (0 for negative, 1 for positive)

# Define features (X) and target (y)
X = data['text']
y = data['sentiment']

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Create a TfidfVectorizer to convert text data into numerical features
vectorizer = TfidfVectorizer()

# Fit the vectorizer to the training data and transform both training and testing data
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

# Convert sparse matrices to dense arrays
X_train_vec = X_train_vec.toarray()
X_test_vec = X_test_vec.toarray()

# Create individual classifiers
clf1 = GaussianNB()
clf2 = SVC(probability=True)  # Set probability=True for soft voting
clf3 = RandomForestClassifier()

# Create a Voting Classifier
# voting='hard' for hard voting, voting='soft' for soft voting
voting_clf = VotingClassifier(estimators=[('nb', clf1), ('svm', clf2), ('rf', clf3)], voting='hard')

# Train the Voting Classifier
voting_clf.fit(X_train_vec, y_train)

# Make predictions on the test set
y_pred = voting_clf.predict(X_test_vec)

# Evaluate the model
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy}")


### # Create a TfidfVectorizer to convert text data into numerical features

from sklearn.feature_extraction.text import TfidfVectorizer

# Sample text data
corpus = [
    'This is the first document.',
    'This document is the second document.',
    'And this is the third one.',
    'Is this the first document?',
]

# Create a TfidfVectorizer object
vectorizer = TfidfVectorizer()

# Fit the vectorizer to the corpus and transform the corpus into a TF-IDF matrix
tfidf_matrix = vectorizer.fit_transform(corpus)

# Print the feature names (words in the vocabulary)
print("Feature names:", vectorizer.get_feature_names_out())

# Print the TF-IDF matrix
print("\nTF-IDF matrix:")
print(tfidf_matrix.toarray())


                                     text  sentiment
0    Excellent quality, highly recommend.          0
1                    I love this product!          0
2    Excellent quality, highly recommend.          0
3                    I love this product!          0
4    Excellent quality, highly recommend.          0
..                                    ...        ...
95         I'm so happy with my purchase!          1
96                   I love this product!          0
97         I'm so happy with my purchase!          1
98                 This is a great movie.          1
99  Fantastic experience, will buy again.          1

[100 rows x 2 columns]
Accuracy: 0.35
Feature names: ['and' 'document' 'first' 'is' 'one' 'second' 'the' 'third' 'this']

TF-IDF matrix:
[[0.         0.46979139 0.58028582 0.38408524 0.         0.
  0.38408524 0.         0.38408524]
 [0.         0.6876236  0.         0.28108867 0.         0.53864762
  0.28108867 0.         0.28108867]
 [0.51184851 0.         0.

In [6]:
#Bagging
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import BaggingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, classification_report
from sklearn.datasets import load_iris

# Load the iris dataset
iris = load_iris()
data = pd.DataFrame(data=np.c_[iris['data'], iris['target']],
                   columns=iris['feature_names'] + ['target'])

# Define features (X) and target (y)
X = data[iris['feature_names']]
y = data['target']

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Create a BaggingClassifier with DecisionTreeClassifier as the base learner
bagging_clf = BaggingClassifier(
    estimator=DecisionTreeClassifier(),    # Changed from base_estimator to estimator
    n_estimators=100,                      # Number of base estimators
    max_samples=0.8,                       # Size of the subsets to use for training each base estimator
    max_features=0.7,                      # Size of the feature subsets
    bootstrap=True,                        # Enable bootstrapping for samples
    bootstrap_features=True,               # Enable bootstrapping for features
    n_jobs=-1,                            # Use all available cores
    random_state=42
)

# Train the BaggingClassifier
bagging_clf.fit(X_train, y_train)

# Make predictions on the test set
y_pred = bagging_clf.predict(X_test)

# Evaluate the model
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy:.4f}")

# Print detailed classification report
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=iris.target_names))

# Feature importance (using the mean feature importance across all trees)
feature_importance = np.mean([
    tree.feature_importances_
    for tree in bagging_clf.estimators_
], axis=0)

# Print feature importance
print("\nFeature Importance:")
for name, importance in zip(iris.feature_names, feature_importance):
    print(f"{name}: {importance:.4f}")

Accuracy: 1.0000

Classification Report:
              precision    recall  f1-score   support

      setosa       1.00      1.00      1.00        10
  versicolor       1.00      1.00      1.00         9
   virginica       1.00      1.00      1.00        11

    accuracy                           1.00        30
   macro avg       1.00      1.00      1.00        30
weighted avg       1.00      1.00      1.00        30


Feature Importance:
sepal length (cm): 0.4809
sepal width (cm): 0.5191
